# Espejo Virtual — prueba de Hunyuan3D-2mv (P-02)

Genera la **forma 3D** de tres marcos a partir de 4 fotos cada uno (frente, costados y atrás) y devuelve un GLB por marco
para cargarlo en el espejo. Es una prueba interna: las fotos son de producto de Edel-Optics y no se publican.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar*.

Después ejecutá cada celda en orden con el botón ▶ (o Shift + Enter) y esperá a que termine antes de pasar a la siguiente.
Tiempo total aproximado: 20–40 minutos, casi todo en las celdas 2 y 5.

**Importante:** si la sesión gratuita queda un rato largo sin actividad, Colab la desconecta y **borra todo** lo instalado
y subido. Si al volver ves errores como `No module named ...`, empezá de nuevo desde la celda 2. Conviene hacer la prueba
de corrido, sin dejar la pestaña abandonada.

## 1. Verificar que hay GPU
Tiene que aparecer una **Tesla T4** (o mejor). Si dice que no hay GPU, revisá el paso de "Antes de empezar".

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Instalar Hunyuan3D-2 (5–10 minutos)
Baja el código oficial de Tencent e instala lo necesario. Solo se usa la parte de **forma** (no la de textura).
Si aparecen avisos en rojo de "dependency conflicts", se pueden ignorar.

**Al terminar, la celda reinicia la sesión sola** (aparece un aviso de que la sesión se cerró o falló: es normal).
Hace falta porque la instalación actualiza librerías que Colab ya tenía cargadas. Lo instalado queda.

**Después de cualquier reinicio, seguí siempre desde la celda 3**: el reinicio borra lo que estaba en memoria.
Si la ejecutás dos veces no pasa nada: si el código ya está bajado, no lo vuelve a bajar.

In [ ]:
%cd /content
![ -d Hunyuan3D-2/hy3dgen ] || git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd /content/Hunyuan3D-2
!pip install -q -r requirements.txt
!pip install -q -e .
# The checkpoint's DINOv2 image encoder uses transformers 4.x parameter names; 5.x renamed them
!pip install -q "transformers>=4.48,<5"
print('Instalación terminada. Reiniciando la sesión...')
import os
os.kill(os.getpid(), 9)   # restart the runtime so updated libraries (e.g. Pillow) load cleanly

## 3. Subir las fotos
Elegí el archivo **`hunyuan-test.zip`** (está en `models-dev/` de la carpeta del proyecto). Si ya lo subiste antes en esta
máquina (por ejemplo, antes de un reinicio), no lo vuelve a pedir. Adentro hay una carpeta por marco
con `d000.webp` (frente), `d090.webp` y `d270.webp` (costados) y `d180.webp` (atrás).

In [ ]:
import zipfile, os
os.chdir('/content')
from google.colab import files

# After a session restart the photos are still on disk: only upload when they are missing
if not os.path.isdir('/content/photos'):
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall('/content/photos')
FRAMES = sorted(d for d in os.listdir('/content/photos') if os.path.isdir(os.path.join('/content/photos', d)))
print('Marcos encontrados:', FRAMES)

## 4. Quitar el fondo blanco y revisar
Hunyuan3D necesita las fotos con **fondo transparente**: si el fondo es blanco opaco, toma la imagen entera como objeto.
Esta celda vuelve transparente todo lo que es casi blanco y muestra el resultado sobre un fondo a cuadros.
**Revisá que el marco se vea entero**, sobre todo las patillas finas del metálico. Si se pierden partes, bajá
`WHITE_THRESHOLD` (por ejemplo a 235) y volvé a ejecutar la celda.

In [ ]:
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

WHITE_THRESHOLD = 245   # 0–255: pixels whose R, G and B are all above this become transparent
VIEWS = {'front': 'd000.webp', 'left': 'd090.webp', 'back': 'd180.webp', 'right': 'd270.webp'}

def remove_white(path):
    rgb = np.asarray(Image.open(path).convert('RGB'))
    background = (rgb >= WHITE_THRESHOLD).all(axis=-1)
    alpha = np.where(background, 0, 255).astype(np.uint8)
    alpha = cv2.dilate(alpha, np.ones((3, 3), np.uint8))   # grow 1 px so thin edges keep their outline
    return Image.fromarray(np.dstack([rgb, alpha]), 'RGBA')

def checker(size, n=24):
    yy, xx = np.indices(size[::-1]) // n
    c = np.where((xx + yy) % 2, 200, 255).astype(np.uint8)
    return Image.fromarray(np.stack([c] * 3, -1)).convert('RGBA')

IMAGES = {}
fig, axes = plt.subplots(len(FRAMES), 4, figsize=(16, 3.2 * len(FRAMES)), squeeze=False)
for r, frame in enumerate(FRAMES):
    IMAGES[frame] = {}
    for c, (view, name) in enumerate(VIEWS.items()):
        img = remove_white(f'/content/photos/{frame}/{name}')
        IMAGES[frame][view] = img
        preview = checker(img.size)
        preview.alpha_composite(img)
        axes[r][c].imshow(preview); axes[r][c].set_title(f'{frame} · {view}'); axes[r][c].axis('off')
plt.tight_layout(); plt.show()

## 5. Cargar el modelo multivista (2–5 minutos la primera vez)
Baja los pesos de `tencent/Hunyuan3D-2mv` desde Hugging Face (unos GB).

In [ ]:
import sys
import importlib
sys.path.insert(0, '/content/Hunyuan3D-2')   # the editable install is not always picked up after the restart
importlib.invalidate_caches()                # in case the path was added before the folder existed
import torch
from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline, FloaterRemover, DegenerateFaceRemover, FaceReducer

pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mv',
    subfolder='hunyuan3d-dit-v2-mv',
    variant='fp16',
)
print('Modelo cargado')

## 6. Generar los modelos (unos minutos por marco)
Para cada marco genera la forma con las 4 vistas, limpia la malla y la guarda como GLB.

El código de Hunyuan define el orden de vistas como "front, front clockwise 90, back, front clockwise 270", pero no queda
claro de qué lado está `left` en nuestras fotos. Por eso, para el **primer marco** genera también una versión con los
costados intercambiados (`_swapped`). Comparando las dos en el paso 7 sabemos cuál es la correcta.

In [ ]:
import time
os.makedirs('/content/out', exist_ok=True)

def generate(images, seed=12345):
    mesh = pipeline(
        image=images,
        num_inference_steps=30,
        octree_resolution=380,
        num_chunks=20000,
        generator=torch.manual_seed(seed),
        output_type='trimesh',
    )[0]
    mesh = FloaterRemover()(mesh)
    mesh = DegenerateFaceRemover()(mesh)
    mesh = FaceReducer()(mesh)
    return mesh

RESULTS = {}
for i, frame in enumerate(FRAMES):
    variants = {'': IMAGES[frame]}
    if i == 0:
        v = IMAGES[frame]
        variants['_swapped'] = {'front': v['front'], 'left': v['right'], 'back': v['back'], 'right': v['left']}
    for suffix, images in variants.items():
        t0 = time.time()
        mesh = generate(images)
        name = f'{frame}{suffix}'
        mesh.export(f'/content/out/{name}.glb')
        RESULTS[name] = mesh
        print(f'{name}: {len(mesh.faces)} caras, {time.time() - t0:.0f} s')

## 7. Vista rápida
Muestra cada malla desde el frente, el costado y arriba. Sirve para ver si la forma tiene sentido y para comparar la versión
`_swapped` del primer marco: la correcta es la que tiene **las patillas derechas y paralelas hacia atrás**.

In [ ]:
fig, axes = plt.subplots(len(RESULTS), 3, figsize=(15, 4 * len(RESULTS)), squeeze=False)
for r, (name, mesh) in enumerate(RESULTS.items()):
    pts = mesh.sample(30000)
    for c, (title, (a, b)) in enumerate({'frente (X, Y)': (0, 1), 'costado (Z, Y)': (2, 1), 'arriba (X, Z)': (0, 2)}.items()):
        axes[r][c].scatter(pts[:, a], pts[:, b], s=0.2, c='k')
        axes[r][c].set_aspect('equal'); axes[r][c].set_title(f'{name} · {title}'); axes[r][c].axis('off')
plt.tight_layout(); plt.show()

## 8. Descargar los resultados
Baja un zip con todos los GLB. Pasáselo a Claude: los carga en el espejo y anota qué tan bien salió cada marco.

In [ ]:
import shutil
shutil.make_archive('/content/hunyuan-results', 'zip', '/content/out')
files.download('/content/hunyuan-results.zip')